# Notebook 01 — Carga y Validación de Datos
**Predicción de Diagnóstico de Cáncer · UAX 2025/2026**

Secciones cubiertas:
- **Sección 0**: Configuración global, seeds y reproducibilidad
- **Sección 1**: Carga individual de los 6 CSVs, validación de integridad y merge

**Artefacto generado:** `data/processed/df_master.parquet`

---
## Sección 0 — Configuración y reproducibilidad

In [1]:
import os, warnings, random
import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')

# ── Seeds ─────────────────────────────────────────────────────────────────────
RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

# ── Rutas ─────────────────────────────────────────────────────────────────────
ROOT      = os.path.abspath('..')          # raíz del proyecto
DATA_RAW  = ROOT                           # CSVs en la raíz
DATA_PROC = os.path.join(ROOT, 'data', 'processed')
OUT_FIGS  = os.path.join(ROOT, 'outputs', 'figures')
OUT_TABS  = os.path.join(ROOT, 'outputs', 'tables')

for path in [DATA_PROC, OUT_FIGS, OUT_TABS]:
    os.makedirs(path, exist_ok=True)

print(f"pandas  {pd.__version__}")
print(f"numpy   {np.__version__}")
print(f"Root:   {ROOT}")

pandas  2.3.3
numpy   2.3.4
Root:   c:\Users\Propietario\Desktop\segundo cuatri\ia\caso cancer


---
## Sección 1 — Carga y validación de datos

Se cargan los **6 CSVs** disponibles. Cada uno representa una colección de la base de
datos documental MongoDB, unidas por `paciente_id`.

> **CASOCANCER_04_ECONOMICOS.csv** está presente pero todas sus variables
> (costes, días de hospitalización) son **consecuencia** del diagnóstico de cáncer,
> no causas → data leakage severo. Se documenta y se excluye del dataset de modelado.

In [2]:
def load_and_validate(path, name, expected_rows=50001):
    """Carga un CSV y verifica integridad básica."""
    df = pd.read_csv(path)
    assert len(df) == expected_rows, \
        f"{name}: esperadas {expected_rows} filas, encontradas {len(df)}"
    assert df['paciente_id'].nunique() == expected_rows, \
        f"{name}: paciente_id no es único"
    assert df.isnull().sum().sum() == 0, \
        f"{name}: contiene {df.isnull().sum().sum()} valores nulos"
    tipos = df.dtypes.to_dict()
    print(f"  {name:<50} shape={str(df.shape):<15} nulos=0  OK")
    return df

In [3]:
print("Cargando y validando CSVs...\n")

bio = load_and_validate(
    os.path.join(DATA_RAW, 'CASOCANCER_01_BIOQUIMICOS.csv'),
    'BIOQUIMICOS  — analítica sanguínea')

cli = load_and_validate(
    os.path.join(DATA_RAW, 'CASOCANCER_02_CLINICOS.csv'),
    'CLINICOS     — historia clínica  [contiene target]')

gen = load_and_validate(
    os.path.join(DATA_RAW, 'CASOCANCER_03_GENETICOS.csv'),
    'GENETICOS    — mutaciones oncogénicas')

eco = load_and_validate(
    os.path.join(DATA_RAW, 'CASOCANCER_04_ECONOMICOS.csv'),
    'ECONOMICOS   — costes [EXCLUIDO: leakage severo]')

hab = load_and_validate(
    os.path.join(DATA_RAW, 'CASOCANCER_05_GENERALES.csv'),
    'GENERALES    — hábitos de vida')

soc = load_and_validate(
    os.path.join(DATA_RAW, 'CASOCANCER_06_SOCIODEMOGRAFICOS.csv'),
    'SOCIODEMOGRAFICOS — perfil social')

print("\nTodos los CSVs cargados y validados correctamente.")

Cargando y validando CSVs...

  BIOQUIMICOS  — analítica sanguínea                 shape=(50001, 8)      nulos=0  OK
  CLINICOS     — historia clínica  [contiene target] shape=(50001, 8)      nulos=0  OK
  GENETICOS    — mutaciones oncogénicas              shape=(50001, 8)      nulos=0  OK
  ECONOMICOS   — costes [EXCLUIDO: leakage severo]   shape=(50001, 6)      nulos=0  OK
  GENERALES    — hábitos de vida                     shape=(50001, 5)      nulos=0  OK
  SOCIODEMOGRAFICOS — perfil social                  shape=(50001, 8)      nulos=0  OK

Todos los CSVs cargados y validados correctamente.


### 1.1 Decisión sobre CASOCANCER_04_ECONOMICOS.csv

Las variables de ese fichero son: `tipo_seguro`, `coste_total`, `coste_farmaco`,
`num_ingresos`, `dias_hospital`. Todas reflejan el estado **posterior** al diagnóstico:
- Un paciente con cáncer genera más ingresos hospitalarios y más coste → si incluyéramos
  estas variables, el modelo aprendería a "predecir" cáncer a partir de sus consecuencias,
  no de sus causas. Esto es **data leakage severo** y haría el modelo inútil en producción.

**Decisión:** se excluye `eco` del merge.

In [4]:
# Merge de los 5 CSVs útiles — eco queda fuera
df = (bio
      .merge(cli, on='paciente_id')
      .merge(gen, on='paciente_id')
      .merge(hab, on='paciente_id')
      .merge(soc, on='paciente_id'))

# Verificaciones post-merge
assert len(df) == 50001,                    "Merge: se han perdido filas"
assert df.isnull().sum().sum() == 0,        "Merge: aparecieron nulos"
assert df['paciente_id'].nunique() == 50001, "Merge: hay duplicados"

print(f"Dataset maestro: {df.shape[0]:,} pacientes  ×  {df.shape[1]} variables")
print(f"Columnas: {list(df.columns)}")

Dataset maestro: 50,001 pacientes  ×  33 variables
Columnas: ['paciente_id', 'glucosa', 'colesterol', 'trigliceridos', 'hemoglobina', 'leucocitos', 'plaquetas', 'creatinina', 'diabetes', 'hipertension', 'obesidad', 'cancer', 'enfermedad_cardiaca', 'asma', 'epoc', 'mut_BRCA1', 'mut_TP53', 'mut_EGFR', 'mut_KRAS', 'mut_PIK3CA', 'mut_ALK', 'mut_BRAF', 'fumador', 'alcohol', 'actividad_fisica', 'vive', 'edad', 'nivel_educativo', 'nivel_ingresos', 'zona', 'estado_civil', 'num_hijos', 'distancia_hospital_km']


### 1.2 Ficha técnica del dataset maestro

In [5]:
ficha = []
for col in df.columns:
    if col == 'paciente_id':
        continue
    dtype = str(df[col].dtype)
    n_unique = df[col].nunique()
    if pd.api.types.is_numeric_dtype(df[col]):
        resumen = f"min={df[col].min():.2f}  max={df[col].max():.2f}  mean={df[col].mean():.2f}"
    else:
        top_vals = df[col].value_counts().head(3).index.tolist()
        resumen = f"categorías: {top_vals}"
    ficha.append({'variable': col, 'dtype': dtype,
                  'n_unique': n_unique, 'resumen': resumen})

df_ficha = pd.DataFrame(ficha)
pd.set_option('display.max_colwidth', 60)
pd.set_option('display.max_rows', 50)
display(df_ficha)

,variable,dtype,n_unique,resumen
0,glucosa,float64,8773,min=55.00 max=179.23 mean=102.19
1,colesterol,float64,13242,min=120.00 max=320.00 mean=193.66
2,trigliceridos,float64,16571,min=50.00 max=321.68 mean=156.23
3,hemoglobina,float64,912,min=8.00 max=18.00 mean=13.93
4,leucocitos,float64,1123,min=2.00 max=15.08 mean=7.15
5,plaquetas,float64,19681,min=100.00 max=489.79 mean=254.97
6,creatinina,float64,161,min=0.35 max=2.10 mean=1.00
7,diabetes,int64,2,min=0.00 max=1.00 mean=0.34
8,hipertension,int64,2,min=0.00 max=1.00 mean=0.44
9,obesidad,int64,2,min=0.00 max=1.00 mean=0.35


### 1.3 Guardado del dataset maestro

In [6]:
out_path = os.path.join(DATA_PROC, 'df_master.parquet')
df.to_parquet(out_path, index=False)
print(f"Guardado: {out_path}")
print(f"Shape: {df.shape}")
print(f"\nListo para el Notebook 02 — EDA y Oracle Score.")

Guardado: c:\Users\Propietario\Desktop\segundo cuatri\ia\caso cancer\data\processed\df_master.parquet
Shape: (50001, 33)

Listo para el Notebook 02 — EDA y Oracle Score.
